# Food Inflation and Commodity Price Analysis
## Somalia Market Intelligence Platform

Comprehensive analysis of food inflation trends, commodity prices, and cost-of-living dynamics across Somalia (2024-2026).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Libraries loaded")

## 1. Data Loading

In [ ]:
# Load datasets
df_food = pd.read_csv('../data/processed/food_prices.csv')
df_col = pd.read_csv('../data/processed/cost_of_living.csv')

df_food['date'] = pd.to_datetime(df_food['date'])
df_col['date'] = pd.to_datetime(df_col['date'])

print(f"📊 Food Prices Dataset:")
print(f"   Records: {len(df_food):,}")
print(f"   Products: {df_food['product_name'].nunique()}")
print(f"   Categories: {df_food['category'].nunique()}")
print(f"\n📊 Cost of Living Dataset:")
print(f"   Records: {len(df_col):,}")
print(f"   Cities: {df_col['city'].nunique()}")

## 2. Top Inflating Products

In [ ]:
# Calculate average inflation rate by product
product_inflation = df_food.groupby('product_name')['inflation_percent'].mean().sort_values(ascending=False)

print("\n📈 Top 10 Most Inflating Products:")
print(product_inflation.head(10))

# Visualization
fig = px.bar(
    x=product_inflation.head(10).values,
    y=product_inflation.head(10).index,
    orientation='h',
    color=product_inflation.head(10).values,
    color_continuous_scale='RdYlGn_r',
    title='Top 10 Most Inflating Food Products',
    labels={'x': 'Average Inflation Rate (%)', 'y': 'Product'},
    height=400
)

fig.update_layout(template='plotly_white', showlegend=False)
fig.write_html('../visuals/top_inflating_products.html')
fig.show()

print("✓ Chart saved")

## 3. Category-Level Inflation Analysis

In [ ]:
# Inflation by category
category_inflation = df_food.groupby('category')['inflation_percent'].agg(['mean', 'std', 'min', 'max']).reset_index()
category_inflation = category_inflation.sort_values('mean', ascending=False)

print("\n📊 Inflation by Category:")
print(category_inflation.to_string(index=False))

# Visualization
fig = go.Figure()

fig.add_trace(go.Bar(
    x=category_inflation['category'],
    y=category_inflation['mean'],
    error_y=dict(type='data', array=category_inflation['std']),
    marker_color=category_inflation['mean'],
    marker_colorscale='Viridis',
    text=category_inflation['mean'].round(2),
    textposition='auto'
))

fig.update_layout(
    title='Average Inflation Rate by Food Category',
    xaxis_title='Category',
    yaxis_title='Inflation Rate (%)',
    height=400,
    template='plotly_white'
)

fig.write_html('../visuals/inflation_by_category.html')
fig.show()

print("✓ Chart saved")

## 4. Regional Inflation Comparison

In [ ]:
# Regional inflation
regional_inflation = df_food.groupby('city')['inflation_percent'].mean().sort_values(ascending=False)

print("\n📍 Inflation Rate by City:")
print(regional_inflation)

# Heatmap of product inflation by city
pivot_inflation = df_food.pivot_table(
    values='inflation_percent',
    index='product_name',
    columns='city',
    aggfunc='mean'
)

fig = go.Figure(data=go.Heatmap(
    z=pivot_inflation.values,
    x=pivot_inflation.columns,
    y=pivot_inflation.index,
    colorscale='RdYlGn_r',
    colorbar=dict(title='Inflation %')
))

fig.update_layout(
    title='Food Inflation Heatmap: Products vs Cities',
    height=600,
    template='plotly_white'
)

fig.write_html('../visuals/inflation_heatmap.html')
fig.show()

print("✓ Heatmap saved")

## 5. Price Trends Over Time

In [ ]:
# Monthly average prices for key staples
df_food['month'] = df_food['date'].dt.to_period('M')

staples = ['Rice', 'Flour', 'Sugar', 'Oil']
staple_data = df_food[df_food['product_name'].isin(staples)]

monthly_prices = staple_data.groupby(['month', 'product_name'])['price_usd'].mean().reset_index()
monthly_prices['month'] = monthly_prices['month'].astype(str)

fig = px.line(
    monthly_prices,
    x='month',
    y='price_usd',
    color='product_name',
    markers=True,
    title='Price Trends for Key Staples',
    labels={'price_usd': 'Price (USD)', 'month': 'Month', 'product_name': 'Product'},
    height=400
)

fig.update_layout(template='plotly_white', hovermode='x unified')
fig.write_html('../visuals/staple_price_trends.html')
fig.show()

print("✓ Price trends chart saved")

## 6. Cost of Living Index Analysis

In [ ]:
# Calculate year-over-year cost of living changes
df_col['year'] = df_col['date'].dt.year
df_col['month'] = df_col['date'].dt.month

# Latest cost of living indices by city
latest_col = df_col.loc[df_col.groupby('city')['date'].idxmax()]
latest_col = latest_col[['city', 'total_cost_index', 'food_index', 'rent_index', 'transport_index']].sort_values('total_cost_index', ascending=False)

print("\n💰 Latest Cost of Living Indices by City:")
print(latest_col.to_string(index=False))

# Visualization - City ranking by cost of living
fig = px.bar(
    latest_col,
    x='total_cost_index',
    y='city',
    orientation='h',
    color='total_cost_index',
    color_continuous_scale='Plasma',
    title='Cost of Living Index by City (Latest)',
    labels={'total_cost_index': 'Cost Index', 'city': 'City'},
    height=400
)

fig.update_layout(template='plotly_white', showlegend=False)
fig.write_html('../visuals/cost_of_living_by_city.html')
fig.show()

print("✓ Chart saved")

## 7. Component Analysis

# Component breakdown of cost of living
latest_col_avg = df_col.loc[df_col['date'] == df_col['date'].max()]
components = ['food_index', 'rent_index', 'transport_index', 'utilities_index', 'internet_index']

city_composition = latest_col_avg[['city'] + components].groupby('city').mean().reset_index()

fig = go.Figure(data=[
    go.Bar(name='Food', x=city_composition['city'], y=city_composition['food_index']),
    go.Bar(name='Rent', x=city_composition['city'], y=city_composition['rent_index']),
    go.Bar(name='Transport', x=city_composition['city'], y=city_composition['transport_index']),
    go.Bar(name='Utilities', x=city_composition['city'], y=city_composition['utilities_index']),
    go.Bar(name='Internet', x=city_composition['city'], y=city_composition['internet_index'])
])

fig.update_layout(
    barmode='stack',
    title='Cost of Living Components by City',
    xaxis_title='City',
    yaxis_title='Index Value',
    height=400,
    template='plotly_white'
)

fig.write_html('../visuals/cost_components_by_city.html')
fig.show()

print("✓ Component analysis chart saved")

## 8. Key Insights and Recommendations

In [ ]:
# Calculate KPIs
top_inflation_product = product_inflation.idxmax()
top_inflation_rate = product_inflation.max()
avg_inflation = df_food['inflation_percent'].mean()
most_expensive_city = regional_inflation.idxmax()
most_affordable_city = regional_inflation.idxmin()

insights = f"""
🇸🇴 FOOD INFLATION & COST OF LIVING ANALYSIS - KEY INSIGHTS

1. INFLATION HOTSPOTS
   • Highest inflating product: {top_inflation_product} ({top_inflation_rate:.2f}% average inflation)
   • Overall food inflation average: {avg_inflation:.2f}%
   • Most expensive city: {most_expensive_city}
   • Most affordable city: {most_affordable_city}

2. PRODUCT CATEGORY INSIGHTS
   • Staple foods (Rice, Flour) show concerning inflation
   • Dairy products (Milk Powder) face supply chain pressures
   • Fresh produce (Banana, Tomato) shows seasonal volatility

3. REGIONAL DISPARITIES
   • Capital city (Mogadishu) has highest cost of living
   • Regional variations reflect security, accessibility, and demand factors
   • Creates arbitrage opportunities for traders

4. HUMANITARIAN IMPLICATIONS
   • High food inflation threatens food security
   • Vulnerable populations disproportionately affected
   • Requires targeted nutrition and livelihood programs

5. BUSINESS OPPORTUNITIES
   • Supply chain optimization between regions
   • Agricultural productivity improvements
   • Storage and preservation infrastructure

6. POLICY RECOMMENDATIONS
   • Implement targeted food subsidies for vulnerable groups
   • Support local production and import substitution
   • Improve regional market integration
   • Monitor inflation monthly for early warning signals
"""

print(insights)

with open('../reports/food_inflation_insights.txt', 'w') as f:
    f.write(insights)

print("\n✓ Insights saved")